In [ ]:
from datasets import load_dataset, DatasetDict

split = load_dataset("mteb/banking77", split="train").train_test_split(test_size=0.1)
test = load_dataset("mteb/banking77", split="test")
ds = DatasetDict({
    "train": split["train"],
    "val": split["test"],
    "test": test,
})
print(ds["train"][:5])
print(ds["train"].features)
print("Range:", min(ds["train"]["label"]), max(ds["train"]["label"]))
print(ds)

In [ ]:
import torch
from transformers import pipeline
from sklearn.metrics import f1_score, classification_report

@torch.no_grad()
def evaluate(model, test_ds, tokenizer):
    model.eval()
    clf = pipeline("text-classification", model=model, tokenizer=tokenizer, device="mps", batch_size=128)
    out = clf(list(test_ds["text"]))
    y_pred = [model.config.label2id[o["label"]] for o in out]
    y_true = list(test_ds["labels"])
    print(classification_report(y_true, y_pred))
    return f1_score(y_true, y_pred, average="macro")

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, set_seed

set_seed(42)
teacher_id = "philschmid/BERT-Banking77"
teacher_tokenizer = AutoTokenizer.from_pretrained(teacher_id)
teacher = AutoModelForSequenceClassification.from_pretrained(teacher_id)
teacher.eval()

In [ ]:
teacher_label2id = {k.lower(): v for k, v in teacher.config.label2id.items()}
dataset_intents = {t.lower() for split in ds.values() for t in split["label_text"]}
assert dataset_intents == set(teacher_label2id), dataset_intents ^ set(teacher_label2id)
ds = ds.map(lambda b: {"labels": [teacher_label2id[t.lower()] for t in b["label_text"]]}, batched=True, remove_columns="label")

In [ ]:
@torch.no_grad()
def teacher_logits(batch):
    teacher.eval()
    enc = teacher_tokenizer(batch["text"], truncation=True, padding=True, return_tensors="pt").to(teacher.device)
    logits = teacher(**enc).logits
    return {"teacher_logits": logits.cpu().numpy()}

for split in ["train", "val"]:
    ds[split] = ds[split].map(teacher_logits, batched=True, batch_size=64)

In [ ]:
print(f"Teacher F1: {evaluate(teacher, ds["test"], teacher_tokenizer)}")

In [ ]:
student_id = "distilbert/distilbert-base-uncased"
student_tokenizer = AutoTokenizer.from_pretrained(student_id)
set_seed(42)
student = AutoModelForSequenceClassification.from_pretrained(student_id, num_labels=77, id2label=teacher.config.id2label, label2id=teacher.config.label2id)

In [ ]:
import torch.nn.functional as F
from transformers import Trainer

class DistillTrainer(Trainer):
    def __init__(self, *args, T, alpha, **kwargs):
        super().__init__(*args, **kwargs)
        self.T = T
        self.alpha = alpha
    
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        teacher_logits = inputs.pop("teacher_logits")
        outputs = model(**inputs)
        student_logits = outputs.logits
        hard = F.cross_entropy(student_logits, labels)
        soft = F.kl_div(F.log_softmax(student_logits / self.T, dim=-1), F.softmax(teacher_logits / self.T, dim=-1), reduction="batchmean") * (self.T ** 2)
        loss = self.alpha*hard + (1-self.alpha)*soft
        return (loss, outputs) if return_outputs else loss

In [ ]:
from transformers import TrainingArguments, EarlyStoppingCallback
import numpy as np

def make_distill_collator(tokenizer):
    def collate(features):
        enc = tokenizer([f["text"] for f in features], truncation=True, padding=True, return_tensors="pt")
        enc["labels"] = torch.tensor([f["labels"] for f in features])
        enc["teacher_logits"] = torch.tensor([f["teacher_logits"] for f in features], dtype=torch.float)
        return enc
    return collate

def compute_metrics(eval_pred):
    logits, labels = eval_pred.predictions, eval_pred.label_ids
    preds = np.argmax(logits, -1)
    return {"f1": f1_score(labels, preds, average="macro")}

trainer = DistillTrainer(
    model=student,
    T=4.0,
    alpha=0.5,
    args=TrainingArguments(
        output_dir="/tmp/results/distill",
        num_train_epochs=15,
        seed=42,
        metric_for_best_model="eval_f1",
        load_best_model_at_end=True,
        greater_is_better=True,
        save_total_limit=2,
        eval_strategy="steps",
        eval_steps=200,
        save_strategy="steps",
        save_steps=200,
        remove_unused_columns=False,
    ),
    train_dataset=ds["train"],
    eval_dataset=ds["val"],
    processing_class=student_tokenizer,
    data_collator=make_distill_collator(student_tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.0)],
)
trainer.train()
print(f"Student F1: {evaluate(student, ds["test"], student_tokenizer)}")

In [ ]:
set_seed(42)
student_hard_label = AutoModelForSequenceClassification.from_pretrained(student_id, num_labels=77, id2label=teacher.config.id2label, label2id=teacher.config.label2id)

In [ ]:
student_hard_label_trainer = DistillTrainer(
    model=student_hard_label,
    T=4.0,
    alpha=1.0,
    args=TrainingArguments(
        output_dir="/tmp/results/distill_hard",
        num_train_epochs=15,
        seed=42,
        metric_for_best_model="eval_f1",
        load_best_model_at_end=True,
        greater_is_better=True,
        save_total_limit=2,
        eval_strategy="steps",
        eval_steps=200,
        save_strategy="steps",
        save_steps=200,
        remove_unused_columns=False,
    ),
    train_dataset=ds["train"],
    eval_dataset=ds["val"],
    processing_class=student_tokenizer,
    data_collator=make_distill_collator(student_tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3, early_stopping_threshold=0.0)],
)
student_hard_label_trainer.train()
print(f"Student Hard Label F1: {evaluate(student_hard_label, ds["test"], student_tokenizer)}")